# Root tip pipeline

The full vision pipeline, from raw plate images to root tips and primary root lengths:

1. Dish crop and padding.
2. U-Net inference with overlapping patches, flip test-time augmentation and hysteresis thresholding.
3. Mask clean-up: micro-closing and gap bridging.
4. Five lanes per plate, then one instance per plant.
5. Skeleton and shortest-path main root extraction.
6. Root length per plant (`main_root_lengths.csv`, Kaggle submission format).
7. Root tip export in robot coordinates (`../data/root_tip_coordinates.json`).

- **Input:** `../models/unet_model_256px.h5` and raw plates in `../data/kaggle_test_images/`.
- **Intermediate masks:** written to folders next to this notebook.

In [ ]:
# 1. SETUP & COMPREHENSIVE INITIALIZATION
import os
import math
from glob import glob

# --- Scientific & Image Stack ---
import cv2
import numpy as np
import scipy.signal
import matplotlib.pyplot as plt

# --- Advanced Morphological Analysis ---
# measure: for connected component properties
# morphology: for skeletonization and noise removal
# filters: for hysteresis thresholding (detecting faint roots)
from skimage import measure, morphology
from skimage.filters import apply_hysteresis_threshold

# --- Graph Theory & Pathfinding ---
# Used for Dijkstra-based main root extraction
import networkx as nx

# --- Deep Learning (TensorFlow/Keras) ---
import tensorflow as tf
import keras.backend as K
from tensorflow.keras.models import load_model

# --- CUSTOM METRIC DEFINITION ---
def f1(y_true, y_pred):
    """
    F1-Score calculation required for model reconstruction.
    Must match the training definition exactly.
    """
    def recall_m(y_true, y_pred):
        TP = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        Positives = K.sum(K.round(K.clip(y_true, 0, 1)))
        return TP / (Positives + K.epsilon())

    def precision_m(y_true, y_pred):
        TP = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        Pred_Positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
        return TP / (Pred_Positives + K.epsilon())

    precision, recall = precision_m(y_true, y_pred), recall_m(y_true, y_pred)
    return 2 * ((precision * recall) / (precision + recall + K.epsilon()))

# --- GLOBAL CONFIGURATION ---
patch_size = 256
model_path = "../models/unet_model_256px.h5"
TEST_DIR = "../data/kaggle_test_images"

# Verify GPU detection
print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU Available: {len(gpus) > 0}")

# Load the model in Inference Mode (compile=False)
if os.path.exists(model_path):
    model = load_model(model_path, custom_objects={"f1": f1}, compile=False)
    print(f"Model loaded: {model_path}")
else:
    print(f"ERROR: Model file not found at {model_path}")

# Discover all test images
test_image_paths = sorted(
    glob(os.path.join(TEST_DIR, "*.png")) +
    glob(os.path.join(TEST_DIR, "*.jpg")) +
    glob(os.path.join(TEST_DIR, "*.tif"))
)
print(f"Found {len(test_image_paths)} images in '{TEST_DIR}'")

In [ ]:
# 2. Quick sanity test on first image (crop + pad preview)
def detect_petri_bbox(gray: np.ndarray) -> tuple[int, int, int]:
    """
    Estimate a square bounding box (x, y, side) around the Petri dish in a grayscale image.

    Uses thresholding + morphology to find candidate contours, then selects the best one
    (large and near the image center). If nothing is found, falls back to a centered square.
    """
    blur = cv2.GaussianBlur(gray, (7, 7), 0)

    thr = cv2.adaptiveThreshold(
        blur, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 51, -10
    )
    if np.mean(thr) < 127:
        thr = 255 - thr

    k = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(thr, cv2.MORPH_OPEN, k, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=1)

    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    h, w = gray.shape

    if not cnts:
        side = min(h, w)
        return ((w - side) // 2, (h - side) // 2, side)

    cx, cy = w / 2.0, h / 2.0

    def score(c):
        x, y, ww, hh = cv2.boundingRect(c)
        area = ww * hh
        mx, my = x + ww / 2.0, y + hh / 2.0
        dist = ((mx - cx) ** 2 + (my - cy) ** 2) ** 0.5
        return area - 0.1 * dist

    best = max(cnts, key=score)
    x, y, ww, hh = cv2.boundingRect(best)

    side = max(ww, hh)
    side = min(side, h, w)
    mx, my = x + ww // 2, y + hh // 2

    xs = int(mx - side / 2)
    ys = int(my - side / 2)

    xs = max(0, min(xs, w - side))
    ys = max(0, min(ys, h - side))

    return xs, ys, side


def refine_to_inner_plate(
    gray: np.ndarray, bbox: tuple[int, int, int], pad: int = 20
) -> tuple[int, int, int]:
    """
    Refine an outer dish bbox to a tighter square bbox around the inner plate region.

    Works by computing row/column intensity profiles in the initial crop, thresholding
    them, finding the longest contiguous "foreground" run in each direction, and then
    centering a square on that region (with optional padding).
    """
    x, y, s = bbox
    crop = gray[y:y + s, x:x + s]

    rmean = crop.mean(axis=1)
    cmean = crop.mean(axis=0)

    _, rbin = cv2.threshold(rmean.astype(np.uint8), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, cbin = cv2.threshold(cmean.astype(np.uint8), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    rbin = rbin > 0
    cbin = cbin > 0

    def longest_run(b):
        best = (0, -1)
        i = 0
        while i < len(b):
            if b[i]:
                j = i
                while j < len(b) and b[j]:
                    j += 1
                if (j - 1) - i > best[1] - best[0]:
                    best = (i, j - 1)
                i = j
            else:
                i += 1
        return best

    r0, r1 = longest_run(rbin)
    c0, c1 = longest_run(cbin)

    r0 = max(0, r0 - pad)
    r1 = min(s - 1, r1 + pad)
    c0 = max(0, c0 - pad)
    c1 = min(s - 1, c1 + pad)

    hh = r1 - r0 + 1
    ww = c1 - c0 + 1
    side = min(hh, ww)

    cy = (r0 + r1) // 2
    cx = (c0 + c1) // 2

    r0 = max(0, cy - side // 2)
    c0 = max(0, cx - side // 2)

    return (x + c0, y + r0, side)


def crop_square(img: np.ndarray, bbox: tuple[int, int, int]) -> np.ndarray:
    """Crop a square region from `img` using bbox = (x, y, side)."""
    x, y, s = bbox
    return img[y:y + s, x:x + s]


def pad_to_patch_size(img: np.ndarray, patch_size: int, value: int = 0):
    """
    Symmetrically pad a 2D image so its height/width are multiples of `patch_size`.

    Returns the padded image plus the (top, bottom) and (left, right) padding sizes.
    """
    h, w = img.shape
    pad_h = (patch_size - (h % patch_size)) % patch_size
    pad_w = (patch_size - (w % patch_size)) % patch_size

    top = pad_h // 2
    bottom = pad_h - top
    left = pad_w // 2
    right = pad_w - left

    padded = cv2.copyMakeBorder(
        img,
        top, bottom,
        left, right,
        cv2.BORDER_CONSTANT,
        value=value
    )
    return padded, (top, bottom), (left, right)


# Quick sanity test on first image
# - Loads the first test image, runs detection + refinement, then pads to patch size.
assert len(test_image_paths) > 0, "No test images found."

p0 = test_image_paths[0]
img0 = cv2.imread(p0, cv2.IMREAD_GRAYSCALE)
assert img0 is not None, f"Could not read: {p0}"

bbox0 = refine_to_inner_plate(img0, detect_petri_bbox(img0), pad=20)
dish0 = crop_square(img0, bbox0)
dish0_padded, pad_h0, pad_w0 = pad_to_patch_size(dish0, patch_size)

print("Example:", os.path.basename(p0))
print("original:", img0.shape)
print("bbox0 (x,y,side):", bbox0)
print("dish:", dish0.shape)
print("padded:", dish0_padded.shape)
print("pad_h (top,bottom):", pad_h0, "pad_w (left,right):", pad_w0)

plt.figure(dpi=200, figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(dish0, cmap="gray"); plt.title("Petri crop"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(dish0_padded, cmap="gray"); plt.title("Padded crop"); plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 3. Visualize all cropped + padded examples (with original sizes)
assert len(test_image_paths) > 0, "No test images found."

pairs = []  # (name, orig_shape, dish, dish_padded, bbox, pad_h, pad_w)

for p in test_image_paths:
    img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print("WARN: Could not read:", p)
        continue

    orig_shape = img.shape  # (H, W)

    bbox = refine_to_inner_plate(img, detect_petri_bbox(img), pad=20)
    dish = crop_square(img, bbox)
    dish_padded, pad_h, pad_w = pad_to_patch_size(dish, patch_size)

    pairs.append((os.path.basename(p), orig_shape, dish, dish_padded, bbox, pad_h, pad_w))

print(f"Prepared {len(pairs)} cropped+padded examples")

# Grid: each row = 1 image, two columns (crop | padded)
rows = len(pairs)
cols = 2
plt.figure(dpi=200, figsize=(12, rows * 3))

for i, (name, orig_shape, dish, dish_padded, bbox, pad_h, pad_w) in enumerate(pairs, start=1):
    H, W = orig_shape

    ax1 = plt.subplot(rows, cols, 2*i - 1)
    ax1.imshow(dish, cmap="gray")
    ax1.set_title(
        f"{name}\norig={H}x{W} | crop={dish.shape[0]}x{dish.shape[1]}\nbbox={bbox}",
        fontsize=8
    )
    ax1.axis("off")

    ax2 = plt.subplot(rows, cols, 2*i)
    ax2.imshow(dish_padded, cmap="gray")
    ax2.set_title(
        f"padded={dish_padded.shape[0]}x{dish_padded.shape[1]}\npad_h={pad_h}, pad_w={pad_w}",
        fontsize=8
    )
    ax2.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 4. Advanced inference (smooth patch stitching + hysteresis thresholding)

def get_spline_window(patch_size, power=2):
    """Create a 2D Hann-based window for smooth overlapping-patch reconstruction."""
    intersection = int(patch_size/2)
    wind_outer = (scipy.signal.windows.hann(patch_size) ** power)
    wind_outer = np.outer(wind_outer, wind_outer) # 2D window
    return wind_outer

def predict_smooth_windowing(img_padded, model, patch_size=256, stride=128, batch_size=32):
    """Predict on all overlapping patches (with flip TTA) and blend back into a full prob map."""
    h, w = img_padded.shape
    spline_window = get_spline_window(patch_size)

    coords = []
    patches = []

    n_h = (h - patch_size) // stride + 1
    n_w = (w - patch_size) // stride + 1

    for i in range(n_h):
        for j in range(n_w):
            y = i * stride
            x = j * stride

            patch = img_padded[y:y+patch_size, x:x+patch_size]

            patch_norm = patch.astype(np.float32) / 255.0
            patch_norm = np.expand_dims(patch_norm, axis=-1) # (256, 256, 1)

            patches.append(patch_norm)
            coords.append((y, x))

    patches_array = np.array(patches)

    patches_flipped = np.fliplr(patches_array)
    full_batch = np.concatenate([patches_array, patches_flipped], axis=0)

    if model.input_shape[-1] == 3:
        full_batch = np.repeat(full_batch, 3, axis=-1)

    print(f"  Predicting {len(full_batch)} patches...")
    all_preds = model.predict(full_batch, batch_size=batch_size, verbose=0)

    n_patches = len(patches)
    preds_orig = all_preds[:n_patches]
    preds_flip = all_preds[n_patches:]

    preds_flip = np.fliplr(preds_flip)
    final_preds = (preds_orig + preds_flip) / 2.0

    prob_map = np.zeros((h, w), dtype=np.float32)
    weight_map = np.zeros((h, w), dtype=np.float32)

    for idx, (y, x) in enumerate(coords):
        pred_patch = final_preds[idx, :, :, 0]
        prob_map[y:y+patch_size, x:x+patch_size] += pred_patch * spline_window
        weight_map[y:y+patch_size, x:x+patch_size] += spline_window

    weight_map[weight_map == 0] = 1.0
    final_prob = prob_map / weight_map

    return final_prob

def apply_double_threshold(prob_map, low=0.05, high=0.25):
    """Apply hysteresis thresholding to convert a prob map into a binary mask."""
    mask = apply_hysteresis_threshold(prob_map, low, high)
    return mask.astype(np.uint8)

# --- EXECUTION LOOP ---
results = []
print(f"Running Advanced Inference on {len(test_image_paths)} images...")

for p in test_image_paths:
    img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print("WARN: Could not read:", p)
        continue

    bbox = detect_petri_bbox(img)
    bbox = refine_to_inner_plate(img, bbox, pad=20)
    petri_dish = crop_square(img, bbox)

    petri_dish_padded, pad_h_tuple, pad_w_tuple = pad_to_patch_size(petri_dish, patch_size)

    pred_prob_padded = predict_smooth_windowing(petri_dish_padded, model, patch_size=256, stride=128)
    pred_bin_padded = apply_double_threshold(pred_prob_padded, low=0.05, high=0.25)

    results.append({
        "path": p,
        "image": img,
        "bbox": bbox,
        "petri_dish": petri_dish,
        "petri_dish_padded": petri_dish_padded,
        "pred_prob_padded": pred_prob_padded,
        "pred_bin_padded": pred_bin_padded,
        "pad_h_tuple": pad_h_tuple,
        "pad_w_tuple": pad_w_tuple
    })
    print(f"Processed {os.path.basename(p)}")

assert len(results) > 0, "Inference completed but produced 0 results (check image paths / reads)."
print(f"Done. Built `results` for {len(results)} images.")

# -----------------------------
# Plot all predictions (grid)
# -----------------------------
cols = 3
rows = math.ceil(len(results) / cols)
fig = plt.figure(dpi=200, figsize=(cols*4, rows*4))

for i, r in enumerate(results, 1):
    name = os.path.basename(r["path"])
    ax = plt.subplot(rows, cols, i)
    ax.imshow(r["pred_bin_padded"], cmap="gray")
    ax.set_title(name, fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

# -------------------------------------
# Side-by-side (crop vs prediction)
# -------------------------------------
cols = 2
rows = len(results)
fig = plt.figure(dpi=200, figsize=(10, rows*3))

for i, r in enumerate(results, 1):
    name = os.path.basename(r["path"])

    ax1 = plt.subplot(rows, 2, 2*i - 1)
    ax1.imshow(r["petri_dish_padded"], cmap="gray")
    ax1.set_title(f"{name} — padded crop", fontsize=9)
    ax1.axis("off")

    ax2 = plt.subplot(rows, 2, 2*i)
    ax2.imshow(r["pred_bin_padded"], cmap="gray")
    ax2.set_title(f"{name} — prediction", fontsize=9)
    ax2.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 5. Save full-size prediction masks (unpad + paste back into original canvas)
OUT_DIR = "test_predictions"
os.makedirs(OUT_DIR, exist_ok=True)

def unpad(pred_padded: np.ndarray, pad_h_tuple, pad_w_tuple):
    """Remove symmetric padding using (top,bottom) and (left,right) pad tuples."""
    (top, bottom) = pad_h_tuple
    (left, right) = pad_w_tuple
    return pred_padded[top: pred_padded.shape[0] - bottom,
                      left: pred_padded.shape[1] - right]

saved_paths = []

for r in results:
    img      = r["image"]                 # original grayscale
    bbox     = r["bbox"]                  # (x,y,side)
    petri    = r["petri_dish"]            # crop (side,side)
    pred_pad = r["pred_bin_padded"]       # (Hpad,Wpad) 0/1
    ph       = r["pad_h_tuple"]
    pw       = r["pad_w_tuple"]

    # 1) UNPAD -> back to crop size
    pred_crop = unpad(pred_pad, ph, pw)
    assert pred_crop.shape == petri.shape, f"Unpad mismatch: {pred_crop.shape} vs {petri.shape}"

    # 2) PASTE back into full image canvas
    x, y, side = bbox
    full_mask = np.zeros(img.shape, dtype=np.uint8)
    full_mask[y:y+side, x:x+side] = pred_crop.astype(np.uint8)

    # 3) Convert to 0/255 (deliverable format)
    full_mask_255 = (full_mask * 255).astype(np.uint8)

    # 4) Save
    base = os.path.splitext(os.path.basename(r["path"]))[0]
    out_path = os.path.join(OUT_DIR, f"{base}_prediction.png")
    cv2.imwrite(out_path, full_mask_255)
    saved_paths.append(out_path)

print("Saved:", len(saved_paths), "masks to:", OUT_DIR)
print("First 5 saved files:", [os.path.basename(p) for p in saved_paths[:5]])

In [ ]:
# 6. Model confidence heatmaps (batch analysis)
# --- ANALYSIS LOOP ---
# Iterate through all processed results to visualize model confidence
print(f"Generating confidence heatmaps for {len(results)} images...")

# Grid Setup: 2 columns per image (Heatmap | Binary)
cols = 2
rows = len(results)

plt.figure(figsize=(14, rows * 5), dpi=150)

for i, r in enumerate(results):
    name = os.path.basename(r["path"])

    # Raw probability map (0.0..1.0) from the model output
    prob_map = r["pred_prob_padded"]

    # Final binary mask after hysteresis thresholding
    bin_mask = r["pred_bin_padded"]

    # 1) HEATMAP: model confidence at each pixel
    ax1 = plt.subplot(rows, cols, 2 * i + 1)
    im = ax1.imshow(prob_map, cmap="jet", vmin=0, vmax=1)
    plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04, label="Confidence")
    ax1.set_title(f"HEATMAP: {name}\n(Raw Model Confidence)", fontsize=10)
    ax1.axis("off")

    # 2) BINARY MASK: final thresholded output
    ax2 = plt.subplot(rows, cols, 2 * i + 2)
    ax2.imshow(bin_mask, cmap="gray")
    ax2.set_title(f"BINARY MASK: {name}\n(Final Thresholded Result)", fontsize=10)
    ax2.axis("off")

plt.tight_layout()
plt.show()

print("Confidence analysis complete.")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import math
from glob import glob
import os

# ──────────────────────────────────────────────
# CONFIGURATION
# ──────────────────────────────────────────────
IMAGE_DIR = "../data/kaggle_test_images"
MASK_DIR = "test_predictions"

OVERLAY_COLOR = (255, 0, 0)  # Red in RGB (matplotlib uses RGB not BGR)
ALPHA = 0.4

# ──────────────────────────────────────────────
# FIND ALL IMAGE/MASK PAIRS
# ──────────────────────────────────────────────
image_paths = sorted(glob(os.path.join(IMAGE_DIR, "test_image_*.png")))
print(f"Found {len(image_paths)} images")

# ──────────────────────────────────────────────
# CREATE AND DISPLAY OVERLAYS
# ──────────────────────────────────────────────
cols = 4
rows = math.ceil(len(image_paths) / cols)

plt.figure(figsize=(cols * 5, rows * 4), dpi=150)

for i, img_path in enumerate(image_paths):
    base = os.path.splitext(os.path.basename(img_path))[0]
    mask_path = os.path.join(MASK_DIR, f"{base}_prediction.png")

    if not os.path.exists(mask_path):
        print(f"WARN: No mask for {base}, skipping")
        continue

    img_gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img_rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask_bool = mask > 127

    overlay = img_rgb.copy()
    overlay[mask_bool] = OVERLAY_COLOR

    result = img_rgb.copy()
    result = cv2.addWeighted(img_rgb, 1 - ALPHA, overlay, ALPHA, 0)
    # Keep original where no mask
    result[~mask_bool] = img_rgb[~mask_bool]

    ax = plt.subplot(rows, cols, i + 1)
    ax.imshow(result)
    ax.set_title(base, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# 7. Batch post-process: micro-close (isotropic) -> bridging (vertical-biased)
# Runs on ALL masks in test_predictions/ and saves to test_predictions_microclose_bridge/

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from skimage import measure, morphology

# --------------------------
# Params (confirmed settings)
# --------------------------
MICRO = dict(
    enabled=True,
    k_size=4,
    iters=1
)

BRIDGE = dict(
    max_dist=60,
    max_dx=4,
    min_dy=8,
    thickness=3,
    enforce_same_lane=True,
    expected_roots=5,
)

# --------------------------
# I/O
# --------------------------
IN_DIR  = "test_predictions"
OUT_DIR = "test_predictions_microclose_bridge"
os.makedirs(OUT_DIR, exist_ok=True)

pred_paths = sorted(glob(os.path.join(IN_DIR, "*_prediction.png")))
assert len(pred_paths) > 0, f"No prediction masks found in: {IN_DIR}"
print(f"Found {len(pred_paths)} masks in {IN_DIR}")

# --------------------------
# Bridging utilities
# --------------------------
def skeleton_endpoints(skel01: np.ndarray) -> np.ndarray:
    """Return (y,x) coordinates of skeleton endpoints for a 0/1 skeleton image."""
    skel = skel01.astype(np.uint8)
    kernel = np.ones((3, 3), dtype=np.uint8)
    neigh = cv2.filter2D(skel, ddepth=-1, kernel=kernel)
    n = neigh - skel
    ys, xs = np.where((skel == 1) & (n == 1))
    return np.stack([ys, xs], axis=1) if len(xs) else np.zeros((0, 2), dtype=np.int32)

def lane_index(x, W, expected_roots):
    """Assign x to a coarse horizontal lane index in [0, expected_roots-1]."""
    bins = np.linspace(0, W, expected_roots + 1)
    lane = np.searchsorted(bins, x, side="right") - 1
    return max(0, min(expected_roots - 1, lane))

def bridge_mask_gaps(mask01: np.ndarray,
                     max_dist=100,
                     max_dx=8,
                     min_dy=6,
                     thickness=1,
                     enforce_same_lane=True,
                     expected_roots=5):
    """Bridge nearby skeleton endpoints across components under simple geometric constraints."""
    mask01 = (mask01 > 0).astype(np.uint8)
    H, W = mask01.shape

    skel = morphology.skeletonize(mask01.astype(bool)).astype(np.uint8)
    lbl = measure.label(mask01, connectivity=2)
    eps = skeleton_endpoints(skel)

    if eps.shape[0] < 2:
        return mask01

    out = mask01.copy()
    used = np.zeros((eps.shape[0],), dtype=bool)

    for i in range(eps.shape[0]):
        if used[i]:
            continue
        y1, x1 = eps[i]
        c1 = lbl[y1, x1]
        if c1 == 0:
            continue

        lane1 = lane_index(x1, W, expected_roots) if enforce_same_lane else None
        best_j = None
        best_d = 1e9

        for j in range(eps.shape[0]):
            if i == j or used[j]:
                continue
            y2, x2 = eps[j]
            c2 = lbl[y2, x2]
            if c2 == 0 or c2 == c1:
                continue  # only connect across components

            dx = abs(int(x2) - int(x1))
            dy = abs(int(y2) - int(y1))
            if dx > max_dx or dy < min_dy:
                continue

            if enforce_same_lane:
                lane2 = lane_index(x2, W, expected_roots)
                if lane2 != lane1:
                    continue

            d = (dx*dx + dy*dy) ** 0.5
            if d <= max_dist and d < best_d:
                best_d = d
                best_j = j

        if best_j is not None:
            y2, x2 = eps[best_j]
            cv2.line(out, (int(x1), int(y1)), (int(x2), int(y2)), color=1, thickness=thickness)
            used[i] = True
            used[best_j] = True

    return out

# --------------------------
# Kernel for micro-closing
# --------------------------
k_close = None
if MICRO["enabled"]:
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (MICRO["k_size"], MICRO["k_size"]))

# --------------------------
# Batch run: micro-close then bridge
# --------------------------
processed = []  # (base, before01, after01, micro01, bridge_only01)

for pred_path in pred_paths:
    m255 = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
    if m255 is None:
        print("WARN: could not read:", pred_path)
        continue

    before01 = (m255 > 127).astype(np.uint8)

    # 1) Micro-close (tiny gaps)
    micro01 = before01.copy()
    if MICRO["enabled"]:
        micro01 = cv2.morphologyEx(micro01, cv2.MORPH_CLOSE, k_close, iterations=MICRO["iters"])

    # 2) Bridging (bigger gaps)
    after01 = bridge_mask_gaps(micro01, **BRIDGE)

    # Diagnostics: pixels added by bridging vs by micro-close
    bridge_only01 = ((after01 > 0) & (micro01 == 0)).astype(np.uint8)

    base = os.path.basename(pred_path).replace("_prediction.png", "")
    out_path = os.path.join(OUT_DIR, f"{base}_prediction_microclose_bridge.png")
    cv2.imwrite(out_path, (after01 * 255).astype(np.uint8))

    processed.append((base, before01, micro01, after01, bridge_only01))

print(f"Saved {len(processed)} masks → {OUT_DIR}")
print(f"MICRO: ellipse {MICRO['k_size']}x{MICRO['k_size']} iters={MICRO['iters']}")
print(f"BRIDGE: max_dist={BRIDGE['max_dist']} max_dx={BRIDGE['max_dx']} min_dy={BRIDGE['min_dy']} thickness={BRIDGE['thickness']}")

# --------------------------
# BIG visualization for ALL images
# Each row: before | after | added-by-bridging
# --------------------------
rows = len(processed)
plt.figure(dpi=220, figsize=(26, max(8, rows * 7.0)))

for i, (base, before01, micro01, after01, bridge_only01) in enumerate(processed, 1):
    ax1 = plt.subplot(rows, 3, 3*i - 2)
    ax1.imshow(before01, cmap="gray")
    ax1.set_title(f"{base}\nBEFORE", fontsize=10)
    ax1.axis("off")

    ax2 = plt.subplot(rows, 3, 3*i - 1)
    ax2.imshow(after01, cmap="gray")
    ax2.set_title("AFTER (micro-close → bridge)", fontsize=10)
    ax2.axis("off")

    ax3 = plt.subplot(rows, 3, 3*i)
    ax3.imshow(bridge_only01, cmap="gray")
    ax3.set_title("ADDED by bridging (after micro-close)", fontsize=10)
    ax3.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 8. Recreate lane_edges.npy (global 5-lane boundaries) from post-processed masks
# Saves to: test_predictions_microclose_bridge/lane_edges.npy

import os
from glob import glob
import cv2
import numpy as np
from skimage import measure

PRED_DIR = "test_predictions_microclose_bridge"  # where you previously saved lane_edges.npy
assert os.path.isdir(PRED_DIR), f"Directory not found: {PRED_DIR}"

# Collect mask images (same directory you used when you printed "Found masks: 19")
mask_paths = sorted(
    glob(os.path.join(PRED_DIR, "*.png")) +
    glob(os.path.join(PRED_DIR, "*.jpg")) +
    glob(os.path.join(PRED_DIR, "*.jpeg")) +
    glob(os.path.join(PRED_DIR, "*.tif")) +
    glob(os.path.join(PRED_DIR, "*.tiff"))
)
assert len(mask_paths) > 0, f"No mask images found in: {PRED_DIR}"

K = 5         # expected_roots
MIN_AREA = 50 # very permissive; we still take top-5 by area per mask

def x_crown_from_region(region):
    """Return crown-x: median x at the component's topmost y (fallback: centroid x)."""
    coords = region.coords  # (N,2) with columns (y,x)
    ymin = coords[:, 0].min()
    xs = coords[coords[:, 0] == ymin][:, 1]
    return float(np.median(xs)) if len(xs) else float(region.centroid[1])

# Accumulate crown-x per lane slot across all masks
slot_xs = [[] for _ in range(K)]

W_ref = None
used = 0

for p in mask_paths:
    m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if m is None:
        continue

    H, W = m.shape
    if W_ref is None:
        W_ref = W
    else:
        # If some masks differ in width, skip them to keep lane edges consistent
        if W != W_ref:
            print(f"WARN: width mismatch for {os.path.basename(p)}: {W} != {W_ref} (skipping)")
            continue

    mask01 = (m > 127).astype(np.uint8)

    labels = measure.label(mask01, connectivity=2)
    regions = measure.regionprops(labels)

    regs = [r for r in regions if r.area >= MIN_AREA]
    if len(regs) < 1:
        continue

    regs = sorted(regs, key=lambda r: r.area, reverse=True)[:max(K, 8)]

    comps = [(r.area, x_crown_from_region(r)) for r in regs]
    comps = sorted(comps, key=lambda t: t[0], reverse=True)[:K]  # top-K by area
    if len(comps) < K:
        # Skip masks with <5 components to keep global lane stats stable
        continue

    # Order left -> right and assign to slot 0..4
    comps_lr = sorted(comps, key=lambda t: t[1])
    for i in range(K):
        slot_xs[i].append(float(comps_lr[i][1]))

    used += 1

assert used > 0, "No usable masks found (did not get >=5 components per mask)."

# Global lane centers = mean crown-x per slot
lane_centers = np.array([np.mean(slot_xs[i]) for i in range(K)], dtype=np.float32)

# Convert centers -> edges via midpoints, with [0, ..., W]
edges = [0.0]
for i in range(K - 1):
    edges.append(float(0.5 * (lane_centers[i] + lane_centers[i + 1])))
edges.append(float(W_ref))
lane_edges = np.array(edges, dtype=np.float32)

out_path = os.path.join(PRED_DIR, "lane_edges.npy")
np.save(out_path, lane_edges)

print(f"Found masks: {len(mask_paths)}")
print(f"Used masks (had >= {K} components): {used}")
print("Global lane centers (px):", lane_centers.tolist())
print("Global lane edges   (px):", lane_edges.tolist())
print("Saved lane edges to:", out_path)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load mask
mask = cv2.imread("test_predictions_microclose_bridge/test_image_02_prediction_microclose_bridge.png", cv2.IMREAD_GRAYSCALE)
lane_edges = np.load("test_predictions_microclose_bridge/lane_edges.npy")

H, W = mask.shape

# Create RGB version
vis = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB)

# Draw lane boundaries (cyan vertical lines)
for edge in lane_edges[1:-1]:
    x = int(edge)
    cv2.line(vis, (x, 0), (x, H), (0, 255, 255), thickness=3)

# Lane labels
for i in range(5):
    left = int(lane_edges[i])
    right = int(lane_edges[i + 1])
    center_x = (left + right) // 2
    cv2.putText(vis, f"Lane {i}", (center_x - 60, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)

# Top exclusion zone (top 15%) — red horizontal line
y_cut = int(0.15 * H)
cv2.line(vis, (0, y_cut), (W, y_cut), (255, 0, 0), thickness=3)
cv2.putText(vis, "Top Exclusion (15%)", (20, y_cut - 15),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 0, 0), 3)

# Top band zone (top 30%) — orange horizontal line
top_band = int(0.30 * H)
cv2.line(vis, (0, top_band), (W, top_band), (255, 165, 0), thickness=3)
cv2.putText(vis, "Top Band (30%)", (20, top_band - 15),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 165, 0), 3)

plt.figure(figsize=(14, 8), dpi=150)
plt.imshow(vis)
plt.title("Lane Boundaries + Cleaning Zones", fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 9. Lane-based cleaning + ordered instance extraction (5 slots, empties allowed)
# Uses data-driven lane edges (lane_edges.npy) + top-anchored lane assignment (x_crown)

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from skimage import measure, morphology

# ----------------------------
# INPUT / OUTPUT
# ----------------------------
IN_DIR  = "test_predictions_microclose_bridge"   # your micro-close + bridging outputs
OUT_DIR = "task6_instances_lane_edges"
os.makedirs(OUT_DIR, exist_ok=True)

lane_edges_path = os.path.join(IN_DIR, "lane_edges.npy")
assert os.path.exists(lane_edges_path), f"lane_edges.npy not found at: {lane_edges_path}"
lane_edges = np.load(lane_edges_path)

mask_paths = sorted(glob(os.path.join(IN_DIR, "*.png")))
# Avoid reading lane_edges.npy as an image
mask_paths = [p for p in mask_paths if not p.endswith(".npy")]

assert len(mask_paths) > 0, f"No masks found in {IN_DIR}"
print(f"Found {len(mask_paths)} masks in {IN_DIR}")
print("Using lane edges:", np.round(lane_edges, 1).tolist())

# ----------------------------
# TUNABLE PARAMETERS
# ----------------------------
CFG = dict(
    expected_roots=5,

    # Global cleanup: ONLY remove true speckles
    speck_area=3,          # keep very small to preserve thin roots/bridges
    exclude_top_frac=0.15,

    # Soft priors: seedlings are near top
    top_band_frac=0.30,    # used only for scoring bonus (not a hard reject)
    top_bonus=250.0,

    # Candidate gating (permissive, but prevents nonsense winning a lane)
    min_area_candidate=10,
    min_height_px=20,      # absolute px; tune upward if noise wins

    # Penalize horizontal-line artifacts (soft)
    horiz_penalty=500.0,
    horiz_aspect=8.0,
    horiz_max_height_px=18,
    horiz_theta_deg=20,

    # Score weights
    w_height=2.0,
    w_major=1.0,
    w_area=0.01,

    # Within-lane merge to absorb detached laterals/fragments
    merge_within_lane=True,
    merge_dist_px=22,      # try 15–35
    merge_min_area=8,

    # Lane emptiness: if best score is too low => empty lane
    min_lane_score=120.0
)

# ----------------------------
# Helpers
# ----------------------------
def lane_from_x(x, lane_edges, expected_roots=5):
    """Map an x-coordinate to a lane index using global lane edges."""
    lane = np.searchsorted(lane_edges, x, side="right") - 1
    return max(0, min(expected_roots - 1, int(lane)))

def x_crown_from_region(region):
    """Crown-x: median x at the component's topmost y (fallback: centroid x)."""
    coords = region.coords  # (N,2) with columns (y,x)
    ymin = coords[:, 0].min()
    xs = coords[coords[:, 0] == ymin][:, 1]
    return float(np.median(xs)) if len(xs) else float(region.centroid[1])

def crop_to_root(mask01, margin=25):
    """Tight crop around nonzero pixels (with margin). Returns None if empty."""
    ys, xs = np.where(mask01 > 0)
    if len(xs) == 0:
        return None
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()
    y0 = max(0, y0 - margin); x0 = max(0, x0 - margin)
    y1 = min(mask01.shape[0]-1, y1 + margin); x1 = min(mask01.shape[1]-1, x1 + margin)
    return mask01[y0:y1+1, x0:x1+1]

def is_horizontal_artifact(region, horiz_aspect, horiz_max_height_px, horiz_theta_deg):
    """Heuristic for thin horizontal components (soft penalty, not a hard reject)."""
    minr, minc, maxr, maxc = region.bbox
    height = maxr - minr
    width  = maxc - minc
    aspect = width / (height + 1e-6)
    theta = abs(np.degrees(region.orientation))  # 0~horizontal in skimage
    return (height <= horiz_max_height_px) and (aspect >= horiz_aspect) and (theta <= horiz_theta_deg)

def lane_clean_and_instances(mask01: np.ndarray, cfg: dict, lane_edges: np.ndarray):
    """
    Select one main component per lane (empties allowed), optionally merge nearby fragments,
    and return both the lane instances and their union.
    """
    mask01 = (mask01 > 0).astype(np.uint8)
    H, W = mask01.shape
    K = cfg["expected_roots"]

    # 1) Speck removal only
    mask01 = morphology.remove_small_objects(mask01.astype(bool), min_size=cfg["speck_area"]).astype(np.uint8)

    labels = measure.label(mask01, connectivity=2)
    regions = measure.regionprops(labels)

    top_band = int(cfg["top_band_frac"] * H)

    # Rule A cutoff: components fully contained above this are considered top-only noise
    y_cut = int(cfg["exclude_top_frac"] * H)

    # 2) Score candidates per lane using x_crown
    lane_cands = [[] for _ in range(K)]  # list of (score, label)
    for r in regions:
        minr, minc, maxr, maxc = r.bbox
        height = maxr - minr

        # Rule A: reject components entirely in the top X% of the image
        if maxr <= y_cut:
            continue

        # permissive gating
        if r.area < cfg["min_area_candidate"] or height < cfg["min_height_px"]:
            continue

        touches_top = (minr <= top_band)
        x_crown = x_crown_from_region(r)
        lane = lane_from_x(x_crown, lane_edges, K)

        score = (
            cfg["w_height"] * float(height) +
            cfg["w_major"]  * float(r.major_axis_length) +
            cfg["w_area"]   * float(r.area)
        )

        if touches_top:
            score += cfg["top_bonus"]

        # penalize horizontal artifacts (soft)
        if (not touches_top) and is_horizontal_artifact(r, cfg["horiz_aspect"], cfg["horiz_max_height_px"], cfg["horiz_theta_deg"]):
            score -= cfg["horiz_penalty"]

        lane_cands[lane].append((score, r.label))

    # 3) Pick best per lane (empties allowed)
    chosen_labels = [None] * K
    chosen_scores = [None] * K

    for lane in range(K):
        if not lane_cands[lane]:
            continue
        lane_cands[lane].sort(key=lambda x: x[0], reverse=True)
        best_score, best_label = lane_cands[lane][0]
        if best_score >= cfg["min_lane_score"]:
            chosen_labels[lane] = best_label
            chosen_scores[lane] = best_score

    # 4) Build lane masks
    lane_masks = []
    for lane in range(K):
        lbl = chosen_labels[lane]
        if lbl is None:
            lane_masks.append(np.zeros_like(mask01, dtype=np.uint8))
        else:
            lane_masks.append((labels == lbl).astype(np.uint8))

    # 5) Merge nearby fragments within same lane (absorbs detached laterals)
    if cfg["merge_within_lane"]:
        # Precompute each region's lane by crown-x
        region_lane = {}
        for r in regions:
            region_lane[r.label] = lane_from_x(x_crown_from_region(r), lane_edges, K)

        for lane in range(K):
            main = lane_masks[lane]
            if main.sum() == 0:
                continue

            inv = ((main == 0) * 255).astype(np.uint8)
            dist = cv2.distanceTransform(inv, cv2.DIST_L2, 3)

            for r in regions:
                if r.label == chosen_labels[lane]:
                    continue
                if r.area < cfg["merge_min_area"]:
                    continue
                if region_lane.get(r.label, -1) != lane:
                    continue

                comp = (labels == r.label)
                ys, xs = np.where(comp)
                if len(xs) == 0:
                    continue

                if float(dist[ys, xs].min()) <= cfg["merge_dist_px"]:
                    main[comp] = 1

            lane_masks[lane] = main.astype(np.uint8)

    clean_union = np.zeros_like(mask01, dtype=np.uint8)
    for m in lane_masks:
        clean_union |= m

    dbg = {
        "top_band_px": top_band,
        "y_cut_px": y_cut,
        "chosen_labels": chosen_labels,
        "chosen_scores": chosen_scores
    }
    return clean_union, lane_masks, dbg

# ----------------------------
# Run on all images + save + BIG visualize
# ----------------------------
processed = []

for mp in mask_paths:
    m255 = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    if m255 is None:
        print("WARN: could not read:", mp)
        continue

    mask01 = (m255 > 127).astype(np.uint8)
    clean01, lane_masks01, dbg = lane_clean_and_instances(mask01, CFG, lane_edges)

    base = os.path.splitext(os.path.basename(mp))[0]

    # Save union clean mask
    out_clean = os.path.join(OUT_DIR, f"{base}_clean.png")
    cv2.imwrite(out_clean, (clean01 * 255).astype(np.uint8))

    # Save lane instances (ordered, empties allowed)
    for i, lm in enumerate(lane_masks01):
        out_p = os.path.join(OUT_DIR, f"{base}_plant_{i:02d}.png")
        cv2.imwrite(out_p, (lm * 255).astype(np.uint8))

    processed.append((base, mask01, clean01, lane_masks01, dbg))

print(f"Saved lane-ordered instances + clean masks → {OUT_DIR}")

# BIG visualization: per image row = input | clean union | 5 lane crops
rows = len(processed)
cols = 2 + CFG["expected_roots"]
plt.figure(dpi=220, figsize=(3.9 * cols, max(10, rows * 3.9)))

for r_i, (base, orig01, clean01, lane_masks01, dbg) in enumerate(processed, 1):
    ax = plt.subplot(rows, cols, (r_i - 1) * cols + 1)
    ax.imshow(orig01, cmap="gray")
    ax.set_title(f"{base}\nINPUT", fontsize=10)
    ax.axis("off")

    ax = plt.subplot(rows, cols, (r_i - 1) * cols + 2)
    ax.imshow(clean01, cmap="gray")
    ax.set_title(f"CLEAN (union)\nscores={dbg['chosen_scores']}", fontsize=9)
    ax.axis("off")

    for lane in range(CFG["expected_roots"]):
        lm = lane_masks01[lane]
        crop = crop_to_root(lm, margin=25)

        ax = plt.subplot(rows, cols, (r_i - 1) * cols + 3 + lane)
        if crop is None:
            ax.imshow(np.zeros((60, 60), dtype=np.uint8), cmap="gray")
            ax.set_title(f"Plant {lane} (EMPTY)", fontsize=9)
        else:
            ax.imshow(crop, cmap="gray")
            ax.set_title(f"Plant {lane}", fontsize=9)
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 10a. Skeletonization and main root extraction (lane instances)
import os
import cv2
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from glob import glob
from skimage import morphology, measure

# ----------------------------
# CONFIGURATION
# ----------------------------
PLANT_DIR = "task6_instances_lane_edges"
TEST_DIR = "../data/kaggle_test_images"
OUT_DIR = "task6_mainroot_simple_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

CFG = dict(
    expected_roots=5,
    crop_margin=50,
    min_component_size=15,

    # Scoring
    height_power=2.0,
    thickness_power=1.5,

    # Pathfinding
    horizontal_penalty=3.0,

    # Standard Start Selection
    start_band=12,
    require_endpoint_start=True,

    # Path Relaxation
    top_relax_band=35,
    top_horizontal_penalty=1.1,

    # --- HUGE PLANT CONFIG ---
    huge_plant_thresh=35000,
    huge_horizontal_penalty=8.0,
    center_bias_strength=2.0,

    # START SELECTION: CENTRAL CORRIDOR STRATEGY
    huge_start_search_band=100,
    huge_corridor_width=15,

    # Path quality scoring for huge plants
    huge_path_sample_depth=100,
    huge_direction_change_penalty=3.0,
    huge_upward_step_penalty=2.0,
    huge_off_center_penalty=0.5,
    huge_spread_penalty=1.0,

    # Small Plant Config
    small_plant_thresh=3050
)

# ----------------------------
# 1. CORE LOGIC
# ----------------------------

def analyze_components(mask01):
    clean = morphology.remove_small_objects(mask01.astype(bool), min_size=CFG['min_component_size']).astype(np.uint8)
    dt = cv2.distanceTransform(clean, cv2.DIST_L2, 3)
    skel = morphology.skeletonize(clean).astype(np.uint8)
    if np.sum(skel) < 5: return None, None

    labels = measure.label(skel, connectivity=2)
    best_label = -1
    best_score = -float('inf')

    for label_id in range(1, labels.max() + 1):
        ys, xs = np.where(labels == label_id)

        min_y, max_y = np.min(ys), np.max(ys)
        height = max_y - min_y
        width = np.max(xs) - np.min(xs)
        avg_thickness = np.mean(dt[ys, xs])

        score = (height ** CFG['height_power']) * (avg_thickness ** CFG['thickness_power'])
        if width > height: score *= 0.1

        if score > best_score:
            best_score = score
            best_label = label_id

    if best_label == -1: return None, None
    winner_mask = (labels == best_label).astype(np.uint8)
    return winner_mask, dt


def score_start_candidate(G, start_node, end_node, dt, plant_center_x):
    """
    Score a start candidate by how straight and central its initial path is.
    Lower score = better (straighter, more central).
    Only used for huge plants (>35k pixels).
    """
    try:
        path = nx.shortest_path(G, source=start_node, target=end_node, weight='weight')
    except:
        return float('inf')

    sample_depth = CFG['huge_path_sample_depth']
    sample = path[:min(sample_depth, len(path))]
    if len(sample) < 10:
        return float('inf')

    xs = [G.nodes[n]['x'] for n in sample]
    ys = [G.nodes[n]['y'] for n in sample]

    # Penalty 1: Direction changes (turns)
    direction_changes = 0
    for i in range(2, len(xs)):
        dx1 = xs[i-1] - xs[i-2]
        dx2 = xs[i] - xs[i-1]
        if (dx1 > 0 and dx2 < 0) or (dx1 < 0 and dx2 > 0):
            direction_changes += 1

    # Penalty 2: Upward movement
    upward_steps = 0
    for i in range(1, len(ys)):
        if ys[i] < ys[i-1]:
            upward_steps += 1

    # Penalty 3: Distance from center
    avg_off_center = np.mean([abs(x - plant_center_x) for x in xs])

    # Penalty 4: Horizontal spread (how much it wanders left-right)
    x_spread = max(xs) - min(xs)

    score = (direction_changes * CFG['huge_direction_change_penalty'] +
             upward_steps * CFG['huge_upward_step_penalty'] +
             avg_off_center * CFG['huge_off_center_penalty'] +
             x_spread * CFG['huge_spread_penalty'])
    return score


def get_path_from_component(winner_mask, dt, plant_area):
    ys, xs = np.where(winner_mask > 0)
    nodes = list(zip(ys, xs))
    node_to_idx = {pt: i for i, pt in enumerate(nodes)}
    min_comp_y = np.min(ys)

    is_huge = plant_area > CFG['huge_plant_thresh']
    plant_center_x = np.median(xs)

    # Graph Construction
    G = nx.Graph()
    for i, (y, x) in enumerate(nodes):
        G.add_node(i, y=y, x=x)
        t1 = dt[y, x]
        dist_from_center_i = 0
        if is_huge: dist_from_center_i = abs(x - plant_center_x) / 50.0

        for dy, dx in [(-1, 0), (-1, 1), (0, 1), (1, 1)]:
            ny, nx_coord = y + dy, x + dx
            if (ny, nx_coord) in node_to_idx:
                j = node_to_idx[(ny, nx_coord)]
                t2 = dt[ny, nx_coord]
                dist = np.sqrt(dy**2 + dx**2)

                dir_penalty = 1.0
                if abs(dx) > abs(dy):
                    dir_penalty = CFG['huge_horizontal_penalty'] if is_huge else CFG['horizontal_penalty']
                    if y <= min_comp_y + CFG['top_relax_band']:
                        dir_penalty = CFG['top_horizontal_penalty']

                avg_thick = (t1 + t2) / 2.0
                thick_factor = 1.0 / (avg_thick + 0.1)

                center_cost = 1.0
                if is_huge:
                    dist_from_center_j = abs(nx_coord - plant_center_x) / 50.0
                    avg_off_center = (dist_from_center_i + dist_from_center_j) / 2.0
                    center_cost = 1.0 + (CFG['center_bias_strength'] * avg_off_center)

                weight = dist * dir_penalty * thick_factor * center_cost
                G.add_edge(i, j, weight=weight)

    # --- START NODE SELECTION ---
    if plant_area < CFG['small_plant_thresh']:
        start_node = min(G.nodes, key=lambda n: G.nodes[n]['y'])

    elif is_huge:
        # Huge Plant: path quality scoring
        candidates = []
        degrees = dict(G.degree())
        for n in G.nodes:
            if degrees[n] == 1:
                y = G.nodes[n]['y']
                if y <= min_comp_y + CFG['huge_start_search_band']:
                    candidates.append(n)
        if not candidates:
            candidates = [n for n in G.nodes if G.nodes[n]['y'] <= min_comp_y + CFG['huge_start_search_band']]

        if candidates:
            # Filter to central corridor first
            central = [n for n in candidates if abs(G.nodes[n]['x'] - plant_center_x) <= CFG['huge_corridor_width']]
            pool = central if central else candidates

            # Get end node for path scoring
            max_y = max(G.nodes[n]['y'] for n in G.nodes)
            ends = [n for n in G.nodes if G.nodes[n]['y'] == max_y]
            end_candidate = ends[0]

            # Score each candidate by path quality
            scored = []
            for n in pool:
                s = score_start_candidate(G, n, end_candidate, dt, plant_center_x)
                scored.append((s, n))

            scored.sort(key=lambda x: x[0])
            start_node = scored[0][1]
        else:
            start_node = min(G.nodes, key=lambda n: G.nodes[n]['y'])

    # CASE C: STANDARD PLANTS [WITH BOTTLENECK FIX]
    else:
        degrees = dict(G.degree())
        start_pool = []
        for n in G.nodes:
            if G.nodes[n]['y'] <= min_comp_y + CFG['start_band']:
                thick = dt[G.nodes[n]['y'], G.nodes[n]['x']]
                deg = degrees[n]
                start_pool.append((n, thick, deg))

        if not start_pool:
            start_pool = [(n, 0, 0) for n in G.nodes if G.nodes[n]['y'] == min_comp_y]

        if CFG['require_endpoint_start']:
            endpoints = [z for z in start_pool if z[2] == 1]
            if len(endpoints) > 0: start_pool = endpoints

        start_pool.sort(key=lambda z: (z[1], z[2]), reverse=True)

        # --- BOTTLENECK CHECK ---
        max_y = max(G.nodes[n]['y'] for n in G.nodes)
        ends = [n for n in G.nodes if G.nodes[n]['y'] == max_y]
        end_node_candidate = ends[0]

        start_node = start_pool[0][0]

        for cand in start_pool[:3]:
            try:
                temp_path = nx.shortest_path(G, source=cand[0], target=end_node_candidate, weight='weight')
                has_bridge = False
                for node_idx in temp_path:
                    y_curr = G.nodes[node_idx]['y']
                    if (min_comp_y + 15) <= y_curr <= (min_comp_y + 25):
                        if dt[y_curr, G.nodes[node_idx]['x']] <= 1.1:
                            has_bridge = True
                            break

                if not has_bridge:
                    start_node = cand[0]
                    break
                else:
                    continue
            except:
                continue

    # --- END NODE SELECTION ---
    max_y = max(G.nodes[n]['y'] for n in G.nodes)
    ends = [n for n in G.nodes if G.nodes[n]['y'] == max_y]
    end_node = ends[0]

    try:
        path_indices = nx.shortest_path(G, source=start_node, target=end_node, weight='weight')
        return [(G.nodes[i]['y'], G.nodes[i]['x']) for i in path_indices]
    except:
        return None

def find_main_root_component_method(mask01):
    plant_area = np.sum(mask01 > 0)
    winner_mask, dt = analyze_components(mask01)
    if winner_mask is None: return None
    return get_path_from_component(winner_mask, dt, plant_area)

# ----------------------------
# 2. VISUALIZATION (lane masks only; no original image canvas)
# ----------------------------
def visualize_final(base, mask_paths, orig_path):
    orig_img = cv2.imread(orig_path) if orig_path else None

    fig = plt.figure(figsize=(20, 10))
    gs = fig.add_gridspec(2, 5)
    global_paths = []

    for lane in range(5):
        ax = fig.add_subplot(gs[0, lane])
        if lane < len(mask_paths) and os.path.exists(mask_paths[lane]):
            mask01 = (cv2.imread(mask_paths[lane], cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
            plant_px = np.sum(mask01 > 0)
            if plant_px > 0:
                path = find_main_root_component_method(mask01)
                vis_mask = cv2.cvtColor(mask01 * 255, cv2.COLOR_GRAY2RGB)
                if path:
                    global_paths.append(path)
                    out_m = np.zeros_like(mask01)
                    for (y, x) in path: out_m[y, x] = 1
                    cv2.imwrite(os.path.join(OUT_DIR, f"{base}_plant_{lane:02d}_main.png"), out_m * 255)
                    pts = np.array([[x, y] for y, x in path], np.int32)
                    cv2.polylines(vis_mask, [pts], False, (255, 0, 0), 2)

                ys, xs = np.where(mask01 > 0)
                if len(ys) > 0:
                    y1, y2 = max(0, ys.min()-50), min(mask01.shape[0], ys.max()+50)
                    x1, x2 = max(0, xs.min()-50), min(mask01.shape[1], xs.max()+50)
                    ax.imshow(vis_mask[y1:y2, x1:x2])
                else:
                    ax.imshow(vis_mask)
                ax.set_title(f"Lane {lane}\nPx: {plant_px}")
            else:
                ax.set_title("Empty")
        else:
            ax.set_title("Missing")
        ax.axis('off')

    # Bottom row: intentionally left blank
    for lane in range(5):
        ax = fig.add_subplot(gs[1, lane])
        ax.axis('off')

    plt.tight_layout()
    plt.show()

# EXECUTION
plant_files = sorted(glob(os.path.join(PLANT_DIR, "*_plant_*.png")))
bases = sorted(list(set([os.path.basename(p).split("_plant_")[0] for p in plant_files])))

print(f"Processing {len(bases)} images with Central Corridor Filter + Path Quality Scoring...")
for base in bases:
    masks = [os.path.join(PLANT_DIR, f"{base}_plant_{i:02d}.png") for i in range(5)]
    hits = glob(os.path.join(TEST_DIR, f"{base}*"))
    orig = hits[0] if hits else None
    visualize_final(base, masks, orig)

In [ ]:
# 10b. Root Tip Identification — Visualize main root + tip location for all images
# For each image, displays 5 plant masks with main root highlighted in red
# and root tip marked with a green circle.

# ----------------------------
# CONFIGURATION
# ----------------------------
PLANT_DIR = "task6_instances_lane_edges"
OUT_DIR = "task6_mainroot_simple_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# Find all unique image bases
plant_files = sorted(glob(os.path.join(PLANT_DIR, "*_plant_*.png")))
bases = sorted(list(set([os.path.basename(p).split("_plant_")[0] for p in plant_files])))

print(f"Found {len(bases)} images to process")
print(f"Plant masks directory: {PLANT_DIR}")

# ----------------------------
# PROCESS AND VISUALIZE ALL IMAGES
# ----------------------------
for img_idx, base in enumerate(bases):
    # Clean base name for display
    display_name = base.replace("_prediction_microclose_bridge", "")

    fig, axes = plt.subplots(1, 5, figsize=(20, 8), dpi=150)

    tips_found = 0

    for lane in range(5):
        mask_path = os.path.join(PLANT_DIR, f"{base}_plant_{lane:02d}.png")
        ax = axes[lane]

        if not os.path.exists(mask_path):
            ax.set_title(f"Lane {lane}\nMissing", fontsize=13, fontweight='bold')
            ax.axis('off')
            continue

        mask01 = (cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
        plant_px = np.sum(mask01 > 0)

        if plant_px == 0:
            ax.set_title(f"Lane {lane}\nEmpty", fontsize=13, fontweight='bold')
            ax.axis('off')
            continue

        # Find main root path
        path = find_main_root_component_method(mask01)

        # Create RGB visualization
        vis = cv2.cvtColor(mask01 * 255, cv2.COLOR_GRAY2RGB)

        if path:
            # Draw main root in red
            pts = np.array([[x, y] for y, x in path], np.int32)
            cv2.polylines(vis, [pts], False, (255, 0, 0), 2)

            # Root tip = last point in path
            tip_y, tip_x = path[-1]

            # Draw green circle around tip
            cv2.circle(vis, (tip_x, tip_y), 25, (0, 255, 0), 3)

            # Draw green dot at center
            cv2.circle(vis, (tip_x, tip_y), 4, (0, 255, 0), -1)

            tips_found += 1
            ax.set_title(f"Lane {lane}\nTip: ({tip_x}, {tip_y})", fontsize=13, fontweight='bold')
        else:
            ax.set_title(f"Lane {lane}\nNo path found", fontsize=13, fontweight='bold')

        # Crop to root region
        ys, xs = np.where(mask01 > 0)
        if len(ys) > 0:
            y1 = max(0, ys.min() - 50)
            y2 = min(mask01.shape[0], ys.max() + 50)
            x1 = max(0, xs.min() - 50)
            x2 = min(mask01.shape[1], xs.max() + 50)
            ax.imshow(vis[y1:y2, x1:x2])
        else:
            ax.imshow(vis)

        ax.axis('off')

    plt.suptitle(f"{display_name} — Root Tips Identified: {tips_found}/5",
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    print(f"[{img_idx+1}/{len(bases)}] {display_name}: {tips_found}/5 tips found")

print(f"\nDone. Processed {len(bases)} images.")

In [ ]:
import os
from glob import glob
import cv2
import numpy as np
import pandas as pd

# ===============================
# CONFIGURATION
# ===============================
PLANT_DIR = "task6_instances_lane_edges"
MAINROOT_DIR = "task6_mainroot_simple_outputs"
OUT_CSV = "main_root_lengths.csv"

# Verify directories exist
assert os.path.isdir(PLANT_DIR), f"Plant directory not found: {PLANT_DIR}"
assert os.path.isdir(MAINROOT_DIR), f"Main-root directory not found: {MAINROOT_DIR}"

def normalize_base(base: str) -> str:
    """Cleans up the base filename to match Kaggle ID format."""
    # Remove known pipeline prefixes
    for bad in [
        "prediction_microclose_bridge",
        "test_predictions_microclose_bridge",
        "predictions_microclose_bridge",
        "microclose_bridge",
    ]:
        base = base.replace(bad, "")
    
    # Clean leftover separators
    base = base.strip("_- ")
    
    # Collapse double underscores
    while "__" in base:
        base = base.replace("__", "_")
        
    return base

def mainroot_length_px(main_img: np.ndarray) -> float:
    """
    Calculates the Euclidean length of a 1-pixel wide skeleton.
    Counts vertical/horizontal steps as 1.0 and diagonal steps as sqrt(2).
    """
    m = (main_img > 127).astype(np.uint8)
    
    # Count connections
    v  = (m[:-1, :]   & m[1:, :]).sum()   # Vertical neighbors
    h  = (m[:, :-1]   & m[:, 1:]).sum()   # Horizontal neighbors
    d1 = (m[:-1, :-1] & m[1:, 1:]).sum()  # Diagonal (Top-Left to Bottom-Right)
    d2 = (m[:-1, 1:]  & m[1:, :-1]).sum() # Diagonal (Top-Right to Bottom-Left)
    
    return float(v + h + np.sqrt(2.0) * (d1 + d2))

# ===============================
# EXECUTION
# ===============================
# 1. Get list of all lane files
lane_files = sorted(glob(os.path.join(PLANT_DIR, "*_plant_*.png")))
assert lane_files, f"No *_plant_*.png found in {PLANT_DIR}"

# 2. Extract unique base names
# Example: 'test_image_01_plant_04.png' -> base: 'test_image_01'
bases = sorted({os.path.basename(p).split("_plant_")[0] for p in lane_files})

K = 5
rows = []

print(f"Found {len(bases)} unique base images. Processing...")

for base in bases:
    base_clean = normalize_base(base) or base
    
    for lane0 in range(K):  # 0 to 4
        # Kaggle IDs are usually 1-based (plant_1 to plant_5)
        plant_id = f"{base_clean}_plant_{lane0+1}"
        
        # --- FIX IS HERE ---
        # Previous code saved as: {base}_plant_{lane:02d}_main.png
        # Your code looked for:   {base}_plant_{lane:02d}_mainroot.png
        filename = f"{base}_plant_{lane0:02d}_main.png"
        main_path = os.path.join(MAINROOT_DIR, filename)
        
        length = 0.0
        if os.path.exists(main_path):
            m = cv2.imread(main_path, cv2.IMREAD_GRAYSCALE)
            if m is not None:
                length = mainroot_length_px(m)
        
        rows.append({"Plant ID": plant_id, "Length (px)": length})

# 3. Save CSV
df = pd.DataFrame(rows).sort_values("Plant ID").reset_index(drop=True)
df.to_csv(OUT_CSV, index=False)

print("Saved:", OUT_CSV)
print(f"Total Rows: {len(df)}")
print(df.head(10))

In [ ]:
# 12. Export root tip coordinates (robot frame) for the robot integration
# ----------------------------
# CONFIGURATION
# ----------------------------
PLANT_DIR = "task6_instances_lane_edges"
OUT_DIR = "../data"  # read by robotics/integration_*.py
os.makedirs(OUT_DIR, exist_ok=True)

# The 3 images we're using for integration (bbox = (779, 78, 2734))
# UPDATE THESE to match your actual filenames
TARGET_IMAGES = [
    "test_image_10",
    "test_image_14",
    "test_image_16",
]

# Plate crop info (from your pipeline Step 2)
BBOX = (779, 78, 2734)  # (x, y, side) - most common bbox

# Plate physical size
PLATE_SIZE_MM = 150.0
SCALE = PLATE_SIZE_MM / BBOX[2]  # mm per pixel

# Plate origin in robot coordinates (meters) — from the integration guide
PLATE_ORIGIN_ROBOT = [0.10775, 0.062, 0.057]  # [x, y, z] in meters
PIPETTE_Z = 0.17  # lowest reachable pipette height (m); the droplet falls to the plate

# Lane edges (paste your actual values)
lane_edges = np.array([0.0, 1430.04, 1980.99, 2494.99, 2955.06, 4202.0], dtype=np.float32)

# ----------------------------
# PROCESS EACH IMAGE
# ----------------------------
all_results = {}

for base in TARGET_IMAGES:
    print(f"\nProcessing: {base}")
    image_tips = []

    for lane in range(5):
        mask_path = os.path.join(PLANT_DIR, f"{base}_prediction_microclose_bridge_plant_{lane:02d}.png")

        if not os.path.exists(mask_path):
            print(f"  Lane {lane}: mask not found, skipping")
            image_tips.append(None)
            continue

        mask01 = (cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
        plant_px = np.sum(mask01 > 0)

        if plant_px == 0:
            print(f"  Lane {lane}: empty mask")
            image_tips.append(None)
            continue

        # Find main root path using your existing function
        path = find_main_root_component_method(mask01)

        if path is None or len(path) == 0:
            print(f"  Lane {lane}: no main root found")
            image_tips.append(None)
            continue

        # Root tip = last point in path (lowest point of main root)
        root_tip_y, root_tip_x = path[-1]

        # Convert from original image space to plate-relative pixels
        plate_px_x = root_tip_x - BBOX[0]
        plate_px_y = root_tip_y - BBOX[1]

        # Convert pixels to mm (relative to plate top-left)
        tip_mm_x = plate_px_x * SCALE
        tip_mm_y = plate_px_y * SCALE

        # Convert to robot coordinates (meters). The plate is rotated -90 degrees in the
        # simulation, so image Y maps to robot X and image X maps to robot Y.
        tip_robot_x = PLATE_ORIGIN_ROBOT[0] + (tip_mm_y / 1000.0)
        tip_robot_y = PLATE_ORIGIN_ROBOT[1] + (tip_mm_x / 1000.0)
        tip_robot_z = PIPETTE_Z

        tip_data = {
            "lane": lane,
            "pixel_original": [int(root_tip_y), int(root_tip_x)],
            "pixel_plate_relative": [round(plate_px_y, 1), round(plate_px_x, 1)],
            "mm_plate_relative": [round(tip_mm_y, 2), round(tip_mm_x, 2)],
            "robot_coords_m": [round(tip_robot_x, 6), round(tip_robot_y, 6), round(tip_robot_z, 6)]
        }

        image_tips.append(tip_data)
        print(f"  Lane {lane}: tip pixel=({root_tip_y}, {root_tip_x}) -> robot=({tip_robot_x:.4f}, {tip_robot_y:.4f}, {tip_robot_z:.4f})")

    all_results[base] = image_tips

# ----------------------------
# SAVE RESULTS
# ----------------------------
import json

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

out_path = os.path.join(OUT_DIR, "root_tip_coordinates.json")
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2, cls=NumpyEncoder)

print(f"\nSaved root tip coordinates to: {out_path}")
print(f"Images processed: {len(all_results)}")
for base, tips in all_results.items():
    valid = sum(1 for t in tips if t is not None)
    print(f"  {base}: {valid}/5 root tips found")